# LOOCV

In [ ]:
loso_feature_columns = [
    "delta_1_seconds",
    "delta_2_seconds",
]

X_loso = (
    delta_train_df[
        loso_feature_columns
    ]
    .to_numpy()
)

y_loso = (
    delta_train_df[
        "class_index"
    ]
    .to_numpy()
)

session_groups = (
    delta_train_df[
        "session"
    ]
    .to_numpy()
)

print("LOSO laps:", len(delta_train_df))
print("LOSO sessions:", sorted(np.unique(session_groups)))
print("Number of sessions:", len(np.unique(session_groups)))
print("Feature shape:", X_loso.shape)

In [ ]:
logo = LeaveOneGroupOut()

loso_predictions = np.full(
    shape=len(delta_train_df),
    fill_value=-1,
    dtype=int,
)

loso_probabilities = np.full(
    shape=(len(delta_train_df), num_classes),
    fill_value=np.nan,
    dtype=float,
)

loso_majority_predictions = np.full(
    shape=len(delta_train_df),
    fill_value=-1,
    dtype=int,
)

fold_records = []

for fold_number, (
    fold_train_indices,
    fold_validation_indices,
) in enumerate(
    logo.split(
        X_loso,
        y_loso,
        groups=session_groups,
    ),
    start=1,
):
    held_out_sessions = np.unique(
        session_groups[fold_validation_indices]
    )

    if len(held_out_sessions) != 1:
        raise ValueError(
            "Each LOSO fold should hold out exactly "
            "one session."
        )

    held_out_session = int(
        held_out_sessions[0]
    )

    X_fold_train = X_loso[
        fold_train_indices
    ]

    y_fold_train = y_loso[
        fold_train_indices
    ]

    X_fold_validation = X_loso[
        fold_validation_indices
    ]

    y_fold_validation = y_loso[
        fold_validation_indices
    ]

    # Logistic regression requires at least two classes.
    training_classes = np.unique(
        y_fold_train
    )

    if len(training_classes) < 2:
        raise ValueError(
            f"Fold holding out session "
            f"{held_out_session} has fewer than "
            "two classes in its training data."
        )

    fold_model = Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=seed,
            ),
        ),
    ])

    fold_model.fit(
        X_fold_train,
        y_fold_train,
    )

    fold_predictions = fold_model.predict(
        X_fold_validation
    )

    fold_model_probabilities = (
        fold_model.predict_proba(
            X_fold_validation
        )
    )

    # LogisticRegression may return only the classes
    # present in that fold's training data. Place each
    # probability in the correct global class column.
    fold_probability_matrix = np.zeros(
        shape=(
            len(fold_validation_indices),
            num_classes,
        ),
        dtype=float,
    )

    fold_class_order = (
        fold_model.named_steps[
            "classifier"
        ].classes_
    )

    for probability_column, class_index in enumerate(
        fold_class_order
    ):
        fold_probability_matrix[
            :,
            int(class_index),
        ] = fold_model_probabilities[
            :,
            probability_column,
        ]

    loso_predictions[
        fold_validation_indices
    ] = fold_predictions

    loso_probabilities[
        fold_validation_indices
    ] = fold_probability_matrix

    # Fair fold-specific majority baseline:
    # determine the majority class from that fold's
    # training data only.
    fold_training_class_counts = np.bincount(
        y_fold_train,
        minlength=num_classes,
    )

    fold_majority_class = int(
        fold_training_class_counts.argmax()
    )

    fold_majority_predictions = np.full(
        shape=len(fold_validation_indices),
        fill_value=fold_majority_class,
        dtype=int,
    )

    loso_majority_predictions[
        fold_validation_indices
    ] = fold_majority_predictions

    fold_accuracy = accuracy_score(
        y_fold_validation,
        fold_predictions,
    )

    fold_majority_accuracy = accuracy_score(
        y_fold_validation,
        fold_majority_predictions,
    )

    fold_records.append({
        "fold": fold_number,
        "held_out_session": held_out_session,
        "training_laps": len(fold_train_indices),
        "held_out_laps": len(
            fold_validation_indices
        ),
        "held_out_classes": len(
            np.unique(y_fold_validation)
        ),
        "fold_accuracy": fold_accuracy,
        "fold_majority_class": (
            class_names[
                fold_majority_class
            ]
        ),
        "fold_majority_accuracy": (
            fold_majority_accuracy
        ),
    })

    print(
        f"Fold {fold_number:02d} | "
        f"Held-out session: {held_out_session:02d} | "
        f"Laps: {len(fold_validation_indices):2d} | "
        f"Accuracy: {fold_accuracy:.4f}"
    )

In [ ]:
assert (
    loso_predictions >= 0
).all(), (
    "At least one training lap did not receive "
    "an out-of-fold prediction."
)

assert (
    loso_majority_predictions >= 0
).all(), (
    "At least one training lap did not receive "
    "a majority-baseline prediction."
)

assert np.isfinite(
    loso_probabilities
).all(), (
    "At least one out-of-fold probability is "
    "missing or non-finite."
)

assert np.allclose(
    loso_probabilities.sum(axis=1),
    1.0,
), (
    "At least one probability row does not sum "
    "to one."
)

print(
    "Every training lap received exactly one "
    "out-of-fold prediction."
)

In [ ]:
loso_accuracy = accuracy_score(
    y_loso,
    loso_predictions,
)

loso_balanced_accuracy = (
    balanced_accuracy_score(
        y_loso,
        loso_predictions,
    )
)

loso_majority_accuracy = accuracy_score(
    y_loso,
    loso_majority_predictions,
)

loso_majority_balanced_accuracy = (
    balanced_accuracy_score(
        y_loso,
        loso_majority_predictions,
    )
)

loso_comparison_df = pd.DataFrame({
    "model": [
        "Fold-specific majority baseline",
        "Delta logistic regression — LOSO",
    ],
    "accuracy": [
        loso_majority_accuracy,
        loso_accuracy,
    ],
    "balanced_accuracy": [
        loso_majority_balanced_accuracy,
        loso_balanced_accuracy,
    ],
})

display(loso_comparison_df)

In [ ]:
print(
    "LOSO delta accuracy:",
    f"{loso_accuracy:.4f}",
)

print(
    "LOSO delta balanced accuracy:",
    f"{loso_balanced_accuracy:.4f}",
)

print(
    "LOSO majority accuracy:",
    f"{loso_majority_accuracy:.4f}",
)

print(
    "LOSO majority balanced accuracy:",
    f"{loso_majority_balanced_accuracy:.4f}",
)

In [ ]:
print(
    classification_report(
        y_loso,
        loso_predictions,
        labels=list(range(num_classes)),
        target_names=class_names,
        zero_division=0,
        digits=4,
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_loso,
    loso_predictions,
    labels=list(range(num_classes)),
    display_labels=class_names,
    values_format="d",
)

plt.title(
    "Delta-Only Logistic Regression\n"
    "Leave-One-Session-Out Predictions"
)

plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
loso_fold_results = pd.DataFrame(
    fold_records
)

display(
    loso_fold_results.sort_values(
        "held_out_session"
    )
)

In [ ]:
loso_results_df = (
    delta_train_df[[
        "global_lap_num",
        "session",
        "session_lap_num",
        "class_name",
        "class_index",
        "delta_1_seconds",
        "delta_2_seconds",
    ]]
    .copy()
)

loso_results_df[
    "loso_predicted_class_index"
] = loso_predictions

loso_results_df[
    "loso_predicted_class_name"
] = [
    class_names[class_index]
    for class_index in loso_predictions
]

loso_results_df[
    "loso_probability_optimal"
] = loso_probabilities[:, 0]

loso_results_df[
    "loso_probability_standard"
] = loso_probabilities[:, 1]

loso_results_df[
    "loso_probability_sub_standard"
] = loso_probabilities[:, 2]

loso_results_df[
    "correct"
] = (
    loso_results_df["class_index"]
    == loso_results_df[
        "loso_predicted_class_index"
    ]
)

display(
    loso_results_df.head(10)
)